# Project 9 — Building a Tiny Autodiff Engine

## Project Description

**Objective**: Understand backpropagation by implementing automatic differentiation for a small computational graph.

**Using Python and NumPy**:

1. Create a simple computational graph supporting operations such as:
   - addition,
   - multiplication,
   - squaring.
2. Perform a forward pass to compute the output.
3. Implement reverse-mode automatic differentiation to compute gradients.
4. Verify your gradients by comparing them with manually derived values.
5. Extend the engine to support a simple activation function (such as ReLU or sigmoid) and verify that gradients still propagate correctly.

The goal is not to build a production framework, but to understand the principles behind the automatic differentiation engines used in PyTorch and other modern deep learning libraries.

## My Solution (from Gemini)

An automatic differentiation (autodiff) engine builds a directed acyclic graph (DAG) during the forward pass to track operations and inputs, then executes a topological reverse pass to apply the chain rule and calculate exact gradients.

In [1]:
import numpy as np

class Value:
    def __init__(self, data, _children=()):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def square(self):
        out = Value(self.data ** 2, (self,))

        def _backward():
            self.grad += (2 * self.data) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(np.maximum(0, self.data), (self,))

        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1.0 / (1.0 + np.exp(-self.data))
        out = Value(s, (self,))

        def _backward():
            self.grad += (s * (1.0 - s)) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # Build topological ordering of all nodes in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # Reverse mode autodiff
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

Consider the function $f(x, y) = \text{ReLU}((x \cdot y + x)^2)$ evaluated at inputs $x = 2.0$ and $y = 3.0$:

In [2]:
# Initialize leaf nodes
x = Value(2.0)
y = Value(3.0)

# Forward pass operations
a = x * y          # a = 2 * 3 = 6
b = a + x          # b = 6 + 2 = 8
c = b.square()     # c = 8^2 = 64
out = c.relu()     # out = max(0, 64) = 64

# Backward pass
out.backward()

print(f"Forward Output: {out.data}")
print(f"Autodiff dx: {x.grad}")
print(f"Autodiff dy: {y.grad}")

Forward Output: 64.0
Autodiff dx: 64.0
Autodiff dy: 32.0


**Manual Gradient Verification**

Analytical derivation using the multivariable chain rule:

For $x = 2$ and $y = 3$, $(x y + x) = 8 > 0$, so $\text{ReLU}$ acts as an identity mapping with a local derivative of 1.$$\frac{\partial f}{\partial x} = \frac{\partial}{\partial x} (x y + x)^2 = 2(x y + x) \cdot (y + 1) = 2(8) \cdot (3 + 1) = 64.0$$

$$\frac{\partial f}{\partial y} = \frac{\partial}{\partial y} (x y + x)^2 = 2(x y + x) \cdot x = 2(8) \cdot 2 = 32.0$$

## What ChatGPT Expected Me to Learn from this Project

The goal of Project 9 was not to build another deep learning framework.

It was to understand how frameworks like PyTorch compute gradients automatically.

By implementing a tiny autodiff engine, you should now appreciate that:

- every operation contributes a local derivative,
- gradients flow backward through the computational graph,
- automatic differentiation is systematic rather than magical.